# Load data

In [1]:
import sys
sys.path.append('../Common')

import CommonYFinance, CommonBinance, CommonMT5, CommonBacktest

In [8]:
symbol = 'VCB.VN'
from_date = '2025-08-01'
to_date = '2025-11-08'
interval = '1d'
data = CommonYFinance.CommonYFinance.loaddataYFinance(symbol, from_date, to_date, interval)

[*********************100%***********************]  1 of 1 completed


In [9]:
data

,Datetime,Adj Close,Close,High,Low,Open,Volume
0,2025-08-01,59763.769531,60200.0,60600.0,59700.0,60300.0,4931693
1,2025-08-04,60657.246094,61100.0,61400.0,60000.0,60100.0,4593017
2,2025-08-05,61054.347656,61500.0,63700.0,61200.0,61500.0,18885850
3,2025-08-06,61848.550781,62300.0,62800.0,61900.0,62100.0,4813719
4,2025-08-07,61252.898438,61700.0,62900.0,61300.0,62900.0,9626430
...,...,...,...,...,...,...,...
64,2025-11-03,59300.000000,59300.0,60300.0,59300.0,60000.0,2857126
65,2025-11-04,60100.000000,60100.0,60400.0,59100.0,59200.0,5171668
66,2025-11-05,60800.000000,60800.0,61100.0,59900.0,60000.0,4607430
67,2025-11-06,60300.000000,60300.0,61300.0,60200.0,60900.0,3175128


In [10]:
# Import các thư viện cần thiết
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Dung ky thuat luu tru MSE vao Dictionary => Sau do check MSE min

In [11]:
data
# Đảm bảo cột 'Datetime' được hiểu là kiểu dữ liệu datetime và đã sắp xếp
data['Datetime'] = pd.to_datetime(data['Datetime'])
data.sort_values(by='Datetime', inplace=True)

In [12]:
data

,Datetime,Adj Close,Close,High,Low,Open,Volume
0,2025-08-01,59763.769531,60200.0,60600.0,59700.0,60300.0,4931693
1,2025-08-04,60657.246094,61100.0,61400.0,60000.0,60100.0,4593017
2,2025-08-05,61054.347656,61500.0,63700.0,61200.0,61500.0,18885850
3,2025-08-06,61848.550781,62300.0,62800.0,61900.0,62100.0,4813719
4,2025-08-07,61252.898438,61700.0,62900.0,61300.0,62900.0,9626430
...,...,...,...,...,...,...,...
64,2025-11-03,59300.000000,59300.0,60300.0,59300.0,60000.0,2857126
65,2025-11-04,60100.000000,60100.0,60400.0,59100.0,59200.0,5171668
66,2025-11-05,60800.000000,60800.0,61100.0,59900.0,60000.0,4607430
67,2025-11-06,60300.000000,60300.0,61300.0,60200.0,60900.0,3175128


In [13]:
# Tạo các biến đặc trưng từ dữ liệu, dua vao cac cot 'Open', 'High', 'Low', 'Volume' manually
features_list = [['Open'], ['Open', 'High'], ['Open', 'High', 'Low'], ['Open', 'High', 'Low', 'Volume']] # List chua cac bien doc lap (fix)

# Khởi tạo biến mục tiêu 'y' là giá đóng cửa => La bien chung ta tim
y = data['Close']

# Tạo một dictionary để lưu trữ MSE cho từng tập hợp các biến đặc trưng
mse_scores = {}

# Lặp qua từng tập hợp các biến đặc trưng và tính toán MSE
for features in features_list: # Chay tung bien doc lap, 4 lan
    X = data[features].copy() # Copy dữ liệu để tránh thay đổi dữ liệu gốc: Lan 1 data[['Open']], lan 2 data[['Open', 'High']], lan 3 data[['Open', 'High', 'Low']], lan 4 data[['Open', 'High', 'Low', 'Volume']]
	# Thay thế giá trị NaN bằng giá trị trung bình của cột tương ứng
    X.fillna(X.mean(), inplace=True) # Mac dinh la khac NA
    
	# Chia dữ liệu thành tập huấn luyện và tập kiểm tra, 80% cho huấn luyện và 20% cho kiểm tra
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train) # Huan luyen mo hinh
    
	# Dự đoán trên tập kiểm tra    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
	# Lưu trữ MSE cho tập hợp các biến đặc trưng
	# Chuyển đổi danh sách các biến đặc trưng thành chuỗi để làm khóa trong dictionary
    mse_scores[str(features)] = mse # Dua vao dictionary mse_scores, key là tên của các features, value là mse

print(mse_scores)

{"['Open']": 3608080.3571428573, "['Open', 'High']": 1214285.7142857143, "['Open', 'High', 'Low']": 1085714.2857142857, "['Open', 'High', 'Low', 'Volume']": 2697142.8571428573}


In [14]:
# Tìm features với MSE thấp nhất trong dictionary
best_features = min(mse_scores, key=mse_scores.get) # Min trong dictionary mse_scores, key là tên của các features
# In ra kết quả
print(f"Best features: {best_features} with MSE: {mse_scores[best_features]}")

Best features: ['Open', 'High', 'Low'] with MSE: 1085714.2857142857


# Dung ky thuat tim ra min MSE

In [15]:
# Đảm bảo cột 'Datetime' được hiểu là kiểu dữ liệu datetime và đã sắp xếp
data['Datetime'] = pd.to_datetime(data['Datetime'])
data.sort_values(by='Datetime', inplace=True)

# Tạo các biến đặc trưng từ dữ liệu, dua vao cac cot 'Open', 'High', 'Low', 'Volume' manually
features_list = [['Open'], ['Open', 'High'], ['Open', 'High', 'Low'], ['Open', 'High', 'Low', 'Volume']]

# Khởi tạo biến mục tiêu 'y' là giá đóng cửa
y = data['Close']

mse_scores = {}

# Dat 1 bien de luu tru gia tri MSE tot nhat
best_mse = float('inf') # Khởi tạo với giá trị vô cực
best_features = None

# Lặp qua từng tập hợp các biến đặc trưng và tính toán MSE
for features in features_list:
    X = data[features].copy() # Copy dữ liệu để tránh thay đổi dữ liệu gốc
	# Thay thế giá trị NaN bằng giá trị trung bình của cột tương ứng
    X.fillna(X.mean(), inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
    # Kiểm tra xem MSE của lần lặp này có nhỏ hơn giá trị tốt nhất hiện tại không
    if mse < best_mse: # Nếu MSE nhỏ hơn giá trị tốt nhất hiện tại thi moi cap nhat
		# Cập nhật giá trị MSE tốt nhất và các biến đặc trưng tương ứng
        best_mse = mse # Set best_mse = mse
        best_features = features # Set best_features = features

print(f"Best features: {best_features} with MSE: {best_mse}")

Best features: ['Open', 'High', 'Low'] with MSE: 1085714.2857142857
